In [44]:
import numpy as np
import pandas as pd

import spice_net as spn

In [45]:
df = pd.read_csv('data/life_expectancy_data.csv')
df = df.dropna()
df = df[df['Status'] == 'Developing']

df['life_expectancy'] = df['Life expectancy '] / df['Life expectancy '].max()
df['adult_mortality'] = df['Adult Mortality'] / df['Adult Mortality'].max()
df['bmi'] = df[' BMI '] / df[' BMI '].max()
df['schooling'] = df['Schooling'] / df['Schooling'].max()
df['total_expenditure'] = df['Total expenditure'] / df['Total expenditure'].max()
df

#df['duration_normed'] = df['Sleep Duration (hours)']

,Country,Year,Status,Life expectancy,Adult Mortality,infant deaths,Alcohol,percentage expenditure,Hepatitis B,Measles,...,Population,thinness 1-19 years,thinness 5-9 years,Income composition of resources,Schooling,life_expectancy,adult_mortality,bmi,schooling,total_expenditure
0,Afghanistan,2015,Developing,65.0,263.0,62,0.01,71.279624,65.0,1154,...,33736494.0,17.2,17.3,0.479,10.1,0.730337,0.363762,0.247730,0.583815,0.567060
1,Afghanistan,2014,Developing,59.9,271.0,64,0.01,73.523582,62.0,492,...,327582.0,17.5,17.5,0.476,10.0,0.673034,0.374827,0.241245,0.578035,0.568450
2,Afghanistan,2013,Developing,59.9,268.0,66,0.01,73.219243,64.0,430,...,31731688.0,17.7,17.7,0.470,9.9,0.673034,0.370678,0.234760,0.572254,0.564976
3,Afghanistan,2012,Developing,59.5,272.0,69,0.01,78.184215,67.0,2787,...,3696958.0,17.9,18.0,0.463,9.8,0.668539,0.376210,0.228275,0.566474,0.592078
4,Afghanistan,2011,Developing,59.2,275.0,71,0.01,7.097109,68.0,3013,...,2978599.0,18.2,18.2,0.454,9.5,0.665169,0.380360,0.223087,0.549133,0.546908
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2933,Zimbabwe,2004,Developing,44.3,723.0,27,4.36,0.000000,68.0,31,...,12777511.0,9.4,9.4,0.407,9.2,0.497753,1.000000,0.351492,0.531792,0.495483
2934,Zimbabwe,2003,Developing,44.5,715.0,26,4.06,0.000000,7.0,998,...,12633897.0,9.8,9.9,0.418,9.5,0.500000,0.988935,0.346304,0.549133,0.453092
2935,Zimbabwe,2002,Developing,44.8,73.0,25,4.43,0.000000,73.0,304,...,125525.0,1.2,1.3,0.427,10.0,0.503371,0.100968,0.341115,0.578035,0.453787
2936,Zimbabwe,2001,Developing,45.3,686.0,25,1.72,0.000000,76.0,529,...,12366165.0,1.6,1.7,0.427,9.8,0.508989,0.948824,0.335927,0.566474,0.428075


In [46]:
som_size = 50

soms = []
soms.append(spn.SpiceNetSom(n_neurons=som_size,
                            key='life_expectancy',
                            value_range_start=df['life_expectancy'].min(),
                            value_range_end=df['life_expectancy'].max(),
                            lrf_tuning_curve=spn.ConstLRF(0.8),
                            lrf_interaction_kernel=spn.ConstLRF(0.8)))

#soms.append(spn.SpiceNetSom(n_neurons=som_size,
#                            key='adult_mortality',
#                            value_range_start=df['adult_mortality'].min(),
#                            value_range_end=df['adult_mortality'].max(),
#                            lrf_tuning_curve=spn.ConstLRF(0.8),
#                            lrf_interaction_kernel=spn.ConstLRF(0.8)))

soms.append(spn.SpiceNetSom(n_neurons=som_size,
                            key='bmi',
                            value_range_start=df['bmi'].min(),
                            value_range_end=df['bmi'].max(),
                            lrf_tuning_curve=spn.ConstLRF(0.8),
                            lrf_interaction_kernel=spn.ConstLRF(0.8)))

soms.append(spn.SpiceNetSom(n_neurons=som_size,
                            key='schooling',
                            value_range_start=df['schooling'].min(),
                            value_range_end=df['schooling'].max(),
                            lrf_tuning_curve=spn.ConstLRF(0.8),
                            lrf_interaction_kernel=spn.ConstLRF(0.8)))

#soms.append(spn.SpiceNetSom(n_neurons=som_size,
#                            key='total_expenditure',
#                            value_range_start=df['total_expenditure'].min(),
#                            value_range_end=df['total_expenditure'].max(),
#                            lrf_tuning_curve=spn.ConstLRF(0.8),
#                            lrf_interaction_kernel=spn.ConstLRF(0.8)))

correlation_matrix = spn.SpiceNetHcm(soms, spn.ConstLRF(0.8), spn.ConstLRF(0.8))

spice_net = spn.SpiceNet(correlation_matrix)

In [47]:
training_data = {
    'life_expectancy': df['life_expectancy'].tolist(),
    #'adult_mortality': df['adult_mortality'].tolist(),
    'bmi': df['bmi'].tolist(),
    'schooling': df['schooling'].tolist(),
    #'total_expenditure': df['total_expenditure'].tolist(),
}
spice_net.fit(training_data, epochs_on_batch=10, batch_size=None, print_output=True)


100%|██████████| 1/1 [00:20<00:00, 20.16s/it]

Time spend on the Components: 
Som: 4.73742151260376 s | Convolution Matrix: 15.420615911483765 s


In [48]:
spn.plot_hcm(correlation_matrix, filter_start=1000)

In [49]:
test_set = df.sample(1000)
errors = []
errors_distance = []
predicted_values = []

for _, row in test_set.iterrows():
    test_data = row
    result_test_value = test_data['life_expectancy']
    predicted = spice_net.decode({
        #'adult_mortality': test_data['adult_mortality'],
        'bmi': test_data['bmi'],
        'schooling': test_data['schooling'],
        #'total_expenditure': test_data['total_expenditure'],
    },
        decoder='bert-optimizer')
    predicted_values.append(predicted)
    errors.append(abs(predicted - result_test_value) / 1 * 100)
    errors_distance.append(abs(predicted - result_test_value))
    print(
        f'Error: {errors[-1]:.6f},\t Predicted: {predicted * df['Life expectancy '].max():.6f},\t Actual: {result_test_value * df['Life expectancy '].max():.6f}')

print(f'Mean Error distance {np.mean(np.array(errors_distance))}')
print(f'Mean Error years {np.mean(np.array(errors_distance)) * df['Life expectancy '].max()}')
print(f'Mean Error in % {np.mean(np.array(errors))}')
print(f'Median Error in % {np.median(np.array(errors))}')
print(f'Error std {np.std(np.array(errors))}')

Error: 2.096159,	 Predicted: 67.634419,	 Actual: 69.500000
Error: 2.662392,	 Predicted: 65.630471,	 Actual: 68.000000
Error: 23.295317,	 Predicted: 54.367168,	 Actual: 75.100000
Error: 1.691222,	 Predicted: 77.394812,	 Actual: 78.900000
Error: 6.260488,	 Predicted: 67.628166,	 Actual: 73.200000
Error: 7.047005,	 Predicted: 67.628166,	 Actual: 73.900000
Error: 13.794007,	 Predicted: 52.823334,	 Actual: 65.100000
Error: 3.073242,	 Predicted: 64.664815,	 Actual: 67.400000
Error: 6.823499,	 Predicted: 80.972914,	 Actual: 74.900000
Error: 5.011380,	 Predicted: 62.560128,	 Actual: 58.100000
Error: 5.370132,	 Predicted: 77.579418,	 Actual: 72.800000
Error: 32.799913,	 Predicted: 50.808078,	 Actual: 80.000000
Error: 10.842616,	 Predicted: 67.449929,	 Actual: 57.800000
Error: 2.535567,	 Predicted: 71.056654,	 Actual: 68.800000
Error: 2.327904,	 Predicted: 67.628166,	 Actual: 69.700000
Error: 13.190003,	 Predicted: 60.560898,	 Actual: 72.300000
Error: 5.951269,	 Predicted: 74.196630,	 Actual: 68

In [50]:
import plotly.express as px

direction = test_set['life_expectancy'].to_numpy() - np.array(predicted_values)
error_df = pd.DataFrame(
    {'Index': range(1, len(errors) + 1),
     'Real': test_set['life_expectancy'],
     'Distance': errors_distance,
     'dir': direction.tolist()})
fig = px.scatter(error_df, x="Index", y="Real", error_y="Distance")
fig.add_scatter(x=error_df['Index'].to_numpy(), y=predicted_values, fillcolor='orange', mode='markers',
                name='Predicted', zorder=1000)
#fig.add_scatter(x=error_df['Index'].to_numpy(), y=error_df['Real'].to_numpy(), fillcolor='green')
fig.show()